# Multi-Query Expansion [Step 3 - Ask It Several Ways, Then Fuse]

> **MLCourse - Agentic AI - Advanced RAG - Query Transformation**

HyDE bets everything on one better query vector. **Multi-query expansion** takes
the opposite bet: instead of finding the single best phrasing, generate
**several** phrasings, retrieve with each, and fuse the result lists.

The insight is about variance rather than bias. Any single phrasing of a
question is a gamble - it may happen to use the same words as the answer, or it
may not. Firing four differently-worded queries and merging their results makes
a single unlucky phrasing much less damaging.

This is also the point where techniques in this course start composing: the
fusion step is exactly the **RRF** algorithm from
[`../01_hybrid_search/03_reciprocal_rank_fusion.ipynb`](../01_hybrid_search/03_reciprocal_rank_fusion.ipynb),
applied to multiple *queries* instead of multiple *retrievers*.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)


def embed(text):
    return encoder.encode([text], normalize_embeddings=True)[0]


def dense_rank(text, top_n=10):
    """Rank paragraph indices by cosine similarity to `text` (best first)."""
    sims = doc_vectors @ embed(text)
    return [int(i) for i in np.argsort(sims)[::-1][:top_n]]


def dense_scores(text):
    return doc_vectors @ embed(text)


print("dense index ready:", doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense index ready: (237, 384)


### 2. Generating query variants

The prompt should push for **genuinely different angles**, not synonym
substitution. Four useful axes:

- different **vocabulary** (formal vs colloquial)
- different **specificity** (narrower and broader)
- different **grammatical form** (question, imperative, keyword phrase)
- different **entry point** (ask about the cause, the effect, the participant)

We parse the model's output as one variant per line - simple, and robust enough
when the prompt asks for exactly that format.

In [5]:
EXPANSION_PROMPT = (
    "You rewrite search queries to improve document retrieval over the novel "
    "'Alice's Adventures in Wonderland'.\n\n"
    "Generate {n} alternative search queries for the question below. Make them "
    "genuinely different from each other: vary the vocabulary, vary how specific "
    "they are, and vary the grammatical form (question, keyword phrase, "
    "statement). Do not number them. Output one query per line and nothing "
    "else.\n\n"
    "Question: {question}"
)


def expand_query(question, n=4):
    raw = ask(EXPANSION_PROMPT.format(n=n, question=question))
    variants = [re.sub(r"^[\-\d\.\)\s]+", "", line).strip()
                for line in raw.splitlines() if line.strip()]
    return [question] + variants[:n]        # always keep the original


QUESTION = "What game does the Queen of Hearts make everyone play?"
queries = expand_query(QUESTION, n=4)

for i, q in enumerate(queries):
    label = "original" if i == 0 else f"variant {i}"
    print(f"{label:>9}: {q}")

 original: What game does the Queen of Hearts make everyone play?
variant 1: Queen of Hearts favorite game
variant 2: cricket match in the Queen's garden
variant 3: What sport is played in the Queen's court?
variant 4: the Queen forces the players to engage in a game of


Keeping the original query in the list is not optional. The rewrites are model
output and can drift; the user's own words are the one query you know is a
faithful statement of the need. If everything else goes wrong, the original
still contributes its results to the fusion.

### 3. Retrieve with every variant

Each query gets its own ranked list. Look at how much - or how little - they
agree.

In [6]:
TOP_N = 10

per_query = {}
for q in queries:
    per_query[q] = dense_rank(q, top_n=TOP_N)

for q, ids in per_query.items():
    print(f"{q[:58]:<60} -> {ids[:5]}")

all_found = set().union(*per_query.values())
original_only = set(per_query[QUESTION])
print(f"\ndocuments the original query alone found in its top {TOP_N}: {len(original_only)}")
print(f"documents the full variant set found              : {len(all_found)}")
print(f"documents ONLY the variants found                 : "
      f"{len(all_found - original_only)}")

What game does the Queen of Hearts make everyone play?       -> [152, 143, 200, 173, 156]
Queen of Hearts favorite game                                -> [200, 143, 152, 149, 172]
cricket match in the Queen's garden                          -> [152, 150, 172, 173, 163]
What sport is played in the Queen's court?                   -> [173, 152, 150, 156, 172]
the Queen forces the players to engage in a game of          -> [152, 173, 172, 163, 150]

documents the original query alone found in its top 10: 10
documents the full variant set found              : 18
documents ONLY the variants found                 : 8


That last number is the recall gain. Those documents were invisible to the
original phrasing and are now candidates. Whether they are *good* candidates is
what the fusion step decides.

### 4. Fusing with RRF

We now merge the ranked lists. RRF is ideal here because it uses only **rank
positions**, which are directly comparable across queries - whereas raw cosine
scores from different query vectors are on subtly different scales and should
not be averaged naively.

```
RRF(d) = sum over queries of  1 / (k + rank of d for that query)
```

A document that appears at rank 3 for three different phrasings beats a document
that appears at rank 1 for one phrasing. That is exactly the robustness we were
buying.

In [7]:
def rrf_fuse(rankings, k=60, top_n=10):
    scores = {}
    for ranked in rankings:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_n]


fused = rrf_fuse(list(per_query.values()), top_n=5)

print("fused top 5 (with the number of variants that found each document):\n")
for rank, (doc_id, score) in enumerate(fused, 1):
    votes = sum(1 for ids in per_query.values() if doc_id in ids)
    baseline_pos = (per_query[QUESTION].index(doc_id) + 1
                    if doc_id in per_query[QUESTION] else None)
    where = f"original rank #{baseline_pos}" if baseline_pos else "NOT found by original"
    print(f"  #{rank} doc_{doc_id}  rrf={score:.4f}  found by {votes}/{len(queries)} "
          f"queries  ({where})")
    print(f"      {paragraphs[doc_id][:110]}...")

fused top 5 (with the number of variants that found each document):

  #1 doc_152  rrf=0.0812  found by 5/5 queries  (original rank #1)
      The players all played at once without waiting for turns, quarrelling all the while, and fighting for the hedg...
  #2 doc_173  rrf=0.0787  found by 5/5 queries  (original rank #4)
      All the time they were playing the Queen never left off quarrelling with the other players, and shouting “Off ...
  #3 doc_150  rrf=0.0777  found by 5/5 queries  (original rank #6)
      “Get to your places!” shouted the Queen in a voice of thunder, and people began running about in all direction...
  #4 doc_172  rrf=0.0774  found by 5/5 queries  (original rank #7)
      The other guests had taken advantage of the Queen’s absence, and were resting in the shade: however, the momen...
  #5 doc_156  rrf=0.0604  found by 4/5 queries  (original rank #5)
      “I don’t think they play at all fairly,” Alice began, in rather a complaining tone, “and they all quarrel so d

### 5. Does it help? Measure it.

Eyeballing is not evidence. We reuse the shared evaluation questions and
keyword-based relevance rules so the comparison is mechanical and unbiased.

In [8]:
# An evaluation question is only useful if we can decide, mechanically and
# without an LLM, whether a retrieved paragraph is relevant. We do that with
# required keyword sets: a paragraph counts as relevant when it contains every
# keyword in at least one of the "any_of" groups. This is a strict, honest,
# reproducible judgement - no LLM grading, no hand-waving.

EVAL_QUESTIONS = [
    {"q": "Why was the White Rabbit in such a hurry?",
     "any_of": [["rabbit", "hurry"], ["rabbit", "late"], ["oh dear", "late"]]},
    {"q": "What happened when Alice drank from the little bottle?",
     "any_of": [["drink", "bottle"], ["bottle", "shutting up like a telescope"],
                ["drank", "telescope"]]},
    {"q": "What game does the Queen of Hearts make everyone play?",
     "any_of": [["croquet"], ["flamingo", "hedgehog"]]},
    {"q": "Who does Alice meet at the mad tea party?",
     "any_of": [["hatter", "dormouse"], ["march hare", "hatter"], ["tea", "dormouse"]]},
    {"q": "What advice does the Caterpillar give Alice?",
     "any_of": [["caterpillar", "mushroom"], ["caterpillar", "keep your temper"],
                ["caterpillar", "who are you"]]},
    {"q": "How does the Cheshire Cat disappear?",
     "any_of": [["grin", "vanish"], ["cheshire cat", "grin"], ["vanished", "grin"]]},
    {"q": "What does the Queen shout whenever she is angry?",
     "any_of": [["off with"], ["queen", "executed"]]},
    {"q": "What happens at the trial of the Knave of Hearts?",
     "any_of": [["knave", "tarts"], ["jury", "verdict"], ["sentence", "verdict"]]},
]


def is_relevant(doc_text, question):
    """True when the paragraph satisfies any one keyword group for the question."""
    low = doc_text.lower()
    return any(all(word in low for word in group) for group in question["any_of"])


def precision_at_k(ranked_ids, question, k=5):
    """Fraction of the top-k retrieved paragraphs that are relevant."""
    top = ranked_ids[:k]
    return sum(is_relevant(paragraphs[i], question) for i in top) / max(len(top), 1)


# Sanity check: every question must have at least one relevant paragraph in
# the corpus, otherwise the metric is meaningless.
for question in EVAL_QUESTIONS:
    n_rel = sum(is_relevant(p, question) for p in paragraphs)
    print(f"{n_rel:3d} relevant paragraphs | {question['q']}")

  7 relevant paragraphs | Why was the White Rabbit in such a hurry?
  3 relevant paragraphs | What happened when Alice drank from the little bottle?
  9 relevant paragraphs | What game does the Queen of Hearts make everyone play?
 10 relevant paragraphs | Who does Alice meet at the mad tea party?
  2 relevant paragraphs | What advice does the Caterpillar give Alice?
  1 relevant paragraphs | How does the Cheshire Cat disappear?
  5 relevant paragraphs | What does the Queen shout whenever she is angry?
  1 relevant paragraphs | What happens at the trial of the Knave of Hearts?


In [9]:
import numpy as np

K = 5
rows = []

for question in EVAL_QUESTIONS:
    q = question["q"]
    base_ids = dense_rank(q, top_n=20)
    variants = expand_query(q, n=3)
    fused_ids = [d for d, _ in rrf_fuse([dense_rank(v, top_n=20) for v in variants],
                                        top_n=20)]
    rows.append((q,
                 precision_at_k(base_ids, question, K),
                 precision_at_k(fused_ids, question, K)))
    print(f"  done: {q[:55]}")
    time.sleep(2.0)         # pace against the 8000 tokens/minute free tier

  done: Why was the White Rabbit in such a hurry?


  done: What happened when Alice drank from the little bottle?


  done: What game does the Queen of Hearts make everyone play?


  done: Who does Alice meet at the mad tea party?


  done: What advice does the Caterpillar give Alice?


  done: How does the Cheshire Cat disappear?


  done: What does the Queen shout whenever she is angry?


  done: What happens at the trial of the Knave of Hearts?


In [10]:
print(f"{'question':<46}{'single':>8}{'multi':>8}{'delta':>8}")
print("-" * 70)
for q, base, multi in rows:
    print(f"{q[:44]:<46}{base:>8.2f}{multi:>8.2f}{multi - base:>+8.2f}")

base_mean = float(np.mean([r[1] for r in rows]))
multi_mean = float(np.mean([r[2] for r in rows]))
print("-" * 70)
print(f"{'MEAN precision@5':<46}{base_mean:>8.3f}{multi_mean:>8.3f}"
      f"{multi_mean - base_mean:>+8.3f}")

question                                        single   multi   delta
----------------------------------------------------------------------
Why was the White Rabbit in such a hurry?         0.40    0.40   +0.00
What happened when Alice drank from the litt      0.40    0.40   +0.00
What game does the Queen of Hearts make ever      0.20    0.20   +0.00
Who does Alice meet at the mad tea party?         0.20    0.60   +0.40
What advice does the Caterpillar give Alice?      0.20    0.20   +0.00
How does the Cheshire Cat disappear?              0.20    0.20   +0.00
What does the Queen shout whenever she is an      0.20    0.40   +0.20
What happens at the trial of the Knave of He      0.20    0.20   +0.00
----------------------------------------------------------------------
MEAN precision@5                                 0.250   0.325  +0.075


Report whatever that says. On a small, clean corpus like this one the gain is
often modest, and sometimes zero - single-query dense retrieval already finds
the obvious paragraphs. Multi-query expansion earns its cost on **large, noisy
corpora** and on **ambiguous questions**, where a single phrasing genuinely
misses whole regions of the index.

### 6. End-to-end generation

Fused context, then a grounded answer from Groq.

In [11]:
def multi_query_rag(question, n_variants=3, top_k=3):
    variants = expand_query(question, n=n_variants)
    fused = rrf_fuse([dense_rank(v, top_n=15) for v in variants], top_n=top_k)
    ids = [d for d, _ in fused]
    context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in ids)
    answer = ask(
        "Answer the question using ONLY the context below. Quote the specific "
        "detail you used. If the context lacks the answer, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return variants, ids, answer


variants, ids, answer = multi_query_rag("Who does Alice meet at the mad tea party?")

print("variants used:")
for v in variants:
    print("  -", v)
print("\nfused context docs:", ids)
print("\nanswer:")
print(answer)

variants used:
  - Who does Alice meet at the mad tea party?
  - Alice encounters Mad Hatter and March Hare during tea
  - characters present at the tea party scene
  - Who are the guests at the tea party?

fused context docs: [137, 128, 133]

answer:
The provided context does not contain a complete list of all characters Alice meets at the mad tea party. It only mentions the **Dormouse** and the **Hatter**.

Specific details quoted:
*   From [doc_137]: "the **Dormouse** fell asleep instantly"
*   From [doc_128]: "said the **Hatter**"


### 7. Pitfalls

- **Cost scales with variants.** N variants means N retrievals and one extra LLM
  call. On Groq's 8000 tokens/minute, five variants per query adds up fast.
  Three or four is the usual sweet spot.
- **Drifting rewrites.** Ask for "different angles" and a model will sometimes
  give you a *different question*. Always keep the original in the pool, and
  consider a cheap guard that drops variants with low similarity to the original.
- **Do not average cosine scores across variants.** Different query vectors have
  different score distributions. Fuse on **ranks** (RRF), not on raw scores.
- **Dedupe before generation.** Fusion frequently surfaces near-identical
  chunks; three copies of one fact crowd out the second fact.
- **Cache aggressively.** Variants for a given query string are stable at
  `temperature=0`. A simple dict keyed on the query removes the cost entirely
  for repeated queries.

### 8. Key takeaways

- Multi-query expansion reduces the **variance** of a single phrasing rather
  than finding one optimal phrasing.
- Always retain the original query alongside the generated variants.
- Fuse with **RRF over ranks**, the same algorithm from `../01_hybrid_search`,
  applied across queries instead of across retrievers.
- Recall gain is real and measurable; precision gain depends on the corpus -
  measure before shipping.

Next: [`04_step_back_prompting.ipynb`](04_step_back_prompting.ipynb) - when the
question is too *specific* for the index to answer directly.